## 02: Preprocessing
Objective: clean the raw data, create new features, and prepare train/test sets for modeling.

### 1. Load Data and Separate the Target

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "../data/titanic_data.csv"
RANDOM_STATE = 42

### [Load ar X/y alada kora]

In [2]:
raw = pd.read_csv(DATA_PATH)
print("Raw shape:", raw.shape)

y = raw["Survived"]
X = raw.drop(columns="Survived")
print("X shape:", X.shape, "| y shape:", y.shape)

Raw shape: (891, 12)
X shape: (891, 11) | y shape: (891,)


## 2. Feature Engineering (row-wise, safe before the split)
#### [Feature function]

In [3]:
TITLE_MAP = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
COMMON_TITLES = ["Mr", "Mrs", "Miss", "Master"]


def add_features(data):
    """Create new features. Every value depends only on the passenger's own row,
    so there is no data leakage even before the train/test split."""
    data = data.copy()

    title = data["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    title = title.replace(TITLE_MAP)
    data["Title"] = title.where(title.isin(COMMON_TITLES), "Rare")

    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
    data["FamilyGroup"] = pd.cut(data["FamilySize"], bins=[0, 1, 4, 11],
                                 labels=["Alone", "Small", "Large"]).astype(str)

    data["has_cabin"] = data["Cabin"].notnull().astype(int)
    return data

##### [Apply & check]

In [4]:
X_fe = add_features(X)

display(X_fe[["Name", "Title", "FamilySize", "IsAlone", "FamilyGroup", "has_cabin"]].head())
print(X_fe["Title"].value_counts())
print(X_fe["FamilyGroup"].value_counts())
print("\nMissing values still left:")
display(X_fe.isnull().sum()[lambda s: s > 0])

,Name,Title,FamilySize,IsAlone,FamilyGroup,has_cabin
0,"Braund, Mr. Owen Harris",Mr,2,0,Small,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,2,0,Small,1
2,"Heikkinen, Miss. Laina",Miss,1,1,Alone,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,2,0,Small,1
4,"Allen, Mr. William Henry",Mr,1,1,Alone,0


Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64
FamilyGroup
Alone    537
Small    292
Large     62
Name: count, dtype: int64

Missing values still left:


Age         177
Cabin       687
Embarked      2
dtype: int64

## 3. Train/Test Split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_fe, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Survival rate in train:", round(y_train.mean() * 100, 1), "%")
print("Survival rate in test: ", round(y_test.mean() * 100, 1), "%")

Train: (712, 16) | Test: (179, 16)
Survival rate in train: 38.3 %
Survival rate in test:  38.5 %


## 4. Fill Missing Values (statistics from the training set only)
#### [Train theke fill value ber kora]

In [6]:
age_median_by_title = X_train.groupby("Title")["Age"].median()
age_median_overall = X_train["Age"].median()
embarked_mode = X_train["Embarked"].mode()[0]

print("Median Age by Title (train only):")
display(age_median_by_title)
print("Overall median Age (fallback):", age_median_overall)
print("Embarked mode:", embarked_mode)

Median Age by Title (train only):


Title
Master     3.0
Miss      22.0
Mr        30.0
Mrs       35.0
Rare      49.0
Name: Age, dtype: float64

Overall median Age (fallback): 28.5
Embarked mode: S


##### [Fill function apply]

In [7]:
def fill_missing(data, age_by_title, age_fallback, embarked_fill):
    """Fill Age and Embarked using values learned from the training set."""
    data = data.copy()
    data["Age"] = data["Age"].fillna(data["Title"].map(age_by_title)).fillna(age_fallback)
    data["Embarked"] = data["Embarked"].fillna(embarked_fill)
    return data


X_train = fill_missing(X_train, age_median_by_title, age_median_overall, embarked_mode)
X_test = fill_missing(X_test, age_median_by_title, age_median_overall, embarked_mode)

##### [Check]

In [8]:
print("Missing in train:")
display(X_train[["Age", "Embarked"]].isnull().sum())
print("Missing in test:")
display(X_test[["Age", "Embarked"]].isnull().sum())

Missing in train:


Age         0
Embarked    0
dtype: int64

Missing in test:


Age         0
Embarked    0
dtype: int64

## 5. Transform and Encode
#### [Transform function]

In [9]:
DROP_COLS = ["PassengerId", "Name", "Ticket", "Cabin", "SibSp", "Parch", "Fare"]
CAT_COLS = ["Sex", "Embarked", "Title", "FamilyGroup"]


def transform(data):
    """Log-transform Fare, drop unused columns, and one-hot encode categories."""
    data = data.copy()
    data["Fare_log"] = np.log1p(data["Fare"])
    data = data.drop(columns=DROP_COLS)
    return pd.get_dummies(data, columns=CAT_COLS, dtype=int)

##### [Apply (test-er column train-er sathe milano)]

In [10]:
X_train_enc = transform(X_train)
X_test_enc = transform(X_test).reindex(columns=X_train_enc.columns, fill_value=0)

print("Train:", X_train_enc.shape, "| Test:", X_test_enc.shape)
print("Same columns:", list(X_train_enc.columns) == list(X_test_enc.columns))
print("Missing values:", X_train_enc.isnull().sum().sum(), X_test_enc.isnull().sum().sum())
print("All numeric:", all(pd.api.types.is_numeric_dtype(t) for t in X_train_enc.dtypes))
print()
print(list(X_train_enc.columns))
X_train_enc.head()

Train: (712, 19) | Test: (179, 19)
Same columns: True
Missing values: 0 0
All numeric: True

['Pclass', 'Age', 'FamilySize', 'IsAlone', 'has_cabin', 'Fare_log', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'FamilyGroup_Alone', 'FamilyGroup_Large', 'FamilyGroup_Small']


,Pclass,Age,FamilySize,IsAlone,has_cabin,Fare_log,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,FamilyGroup_Alone,FamilyGroup_Large,FamilyGroup_Small
692,3,30.0,1,1,0,4.051712,0,1,0,0,1,0,0,1,0,0,1,0,0
481,2,30.0,1,1,0,0.000000,0,1,0,0,1,0,0,1,0,0,1,0,0
527,1,30.0,1,1,1,5.406181,0,1,0,0,1,0,0,1,0,0,1,0,0
855,3,18.0,2,0,0,2.336987,1,0,0,0,1,0,0,0,1,0,0,0,1
801,2,31.0,3,0,0,3.305054,1,0,0,0,1,0,0,0,1,0,0,0,1


## 6. Save Processed Data
###### [Save (fill value-o rakhbo)]

In [11]:
import json
from pathlib import Path

OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_train_enc.to_csv(OUT_DIR / "X_train.csv", index=False)
X_test_enc.to_csv(OUT_DIR / "X_test.csv", index=False)
y_train.to_frame().to_csv(OUT_DIR / "y_train.csv", index=False)
y_test.to_frame().to_csv(OUT_DIR / "y_test.csv", index=False)

fill_values = {
    "age_median_by_title": age_median_by_title.to_dict(),
    "age_median_overall": float(age_median_overall),
    "embarked_mode": embarked_mode,
}
with open(OUT_DIR / "fill_values.json", "w") as f:
    json.dump(fill_values, f, indent=2)

print(sorted(p.name for p in OUT_DIR.iterdir()))

['X_test.csv', 'X_train.csv', 'fill_values.json', 'y_test.csv', 'y_train.csv']


##### [Reload kore verify]

In [12]:
chk_X = pd.read_csv(OUT_DIR / "X_train.csv")
chk_y = pd.read_csv(OUT_DIR / "y_train.csv")
print(chk_X.shape, chk_y.shape, "| rows match:", len(chk_X) == len(chk_y))

(712, 19) (712, 1) | rows match: True


#### I split the data first, and then calculated the fill values from the training set only, so the test set stays unseen. I filled Age by the median of each title, applied the log transform to Fare, and one-hot encoded the categories. The result is 19 numeric features.